# Breast Cancer Detection — Notebook 3 (fast version)
## DenseNet-121 on IDC Histopathology Patches

Invasive Ductal Carcinoma (IDC) detection on 50×50 histopathology patches (~277k patches from 279 patients), with a **patient-wise split** so no patient appears in two splits.

**Fast settings (fits comfortably in one free-Colab session, ~10–15 min):**
- only the sampled patches are extracted from the zip (no full unzip of ~277k files)
- balanced subsample: 3,000 patches/class for training, 750/class for val and test
- 96×96 input (patches are natively 50×50, so 224 adds cost without information)
- 4 epochs, mixed precision (AMP)

### 1. Setup

In [ ]:
import time
t_start = time.time()

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

base_path = "/content/drive/MyDrive/breast_cancer_project"

for folder in ["data", "models", "results", "plots"]:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Project folders ready")

In [ ]:
import os, shutil

# If kaggle.json is in the root of your Google Drive it is used automatically,
# so the notebook needs no further interaction after the Drive mount.
drive_token = "/content/drive/MyDrive/kaggle.json"
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

if os.path.exists(drive_token):
    shutil.copy(drive_token, os.path.expanduser("~/.kaggle/kaggle.json"))
    print("kaggle.json copied from Drive")
else:
    from google.colab import files
    files.upload()   # upload kaggle.json
    shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

### 2. Dataset — IDC breast histopathology (download only, no full unzip)

In [ ]:
!kaggle datasets download -d paultimothymooney/breast-histopathology-images

In [ ]:
import zipfile
import pandas as pd

zf = zipfile.ZipFile("breast-histopathology-images.zip")
names = [n for n in zf.namelist() if n.endswith(".png")]

# the archive stores every patch twice (at the root and under
# IDC_regular_ps50_idx5/); keep a single copy
dup_prefix = "IDC_regular_ps50_idx5/"
root_copies = [n for n in names if not n.startswith(dup_prefix)]
if root_copies:
    names = root_copies

df = pd.DataFrame({"zip_name": names})
df["label"] = df["zip_name"].apply(lambda n: int(n.split("/")[-2]))     # 0 = non-IDC, 1 = IDC
df["patient"] = df["zip_name"].apply(lambda n: n.split("/")[-3])

print("Total patches:", len(df))
print(df["label"].value_counts())
print("Patients:", df["patient"].nunique())

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
from sklearn.model_selection import train_test_split

# -------- patient-wise split (no patient appears in two splits) --------
patients = df["patient"].unique()

train_pat, temp_pat = train_test_split(patients, test_size=0.30, random_state=42)
val_pat, test_pat = train_test_split(temp_pat, test_size=0.50, random_state=42)

def balanced_sample(frame, n_per_class, seed=42):
    return (frame.groupby("label", group_keys=False)
                 .apply(lambda g: g.sample(min(len(g), n_per_class), random_state=seed))
                 .reset_index(drop=True))

train_df = balanced_sample(df[df["patient"].isin(train_pat)], 3000)
val_df = balanced_sample(df[df["patient"].isin(val_pat)], 750)
test_df = balanced_sample(df[df["patient"].isin(test_pat)], 750)

print("Train:", train_df["label"].value_counts().to_dict(), "patients:", len(train_pat))
print("Val:  ", val_df["label"].value_counts().to_dict(), "patients:", len(val_pat))
print("Test: ", test_df["label"].value_counts().to_dict(), "patients:", len(test_pat))

In [ ]:
import os

# extract only the ~9,000 sampled patches
os.makedirs("data", exist_ok=True)
for frame in (train_df, val_df, test_df):
    for n in frame["zip_name"]:
        zf.extract(n, "data")
    frame["path"] = "data/" + frame["zip_name"]

zf.close()
print("Extracted:", len(train_df) + len(val_df) + len(test_df), "patches")

In [ ]:
from torchvision import transforms

IMG_SIZE = 96

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),          # histology has no canonical orientation
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class IDCDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = row["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

train_loader = DataLoader(IDCDataset(train_df, train_transform),
                          batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(IDCDataset(val_df, val_transform),
                        batch_size=128, num_workers=2, pin_memory=True)
test_loader = DataLoader(IDCDataset(test_df, val_transform),
                         batch_size=128, num_workers=2, pin_memory=True)

img, label = IDCDataset(train_df, train_transform)[0]
print(img.shape, label)

In [ ]:
import matplotlib.pyplot as plt

sample = pd.concat([train_df[train_df.label == 0].head(4),
                    train_df[train_df.label == 1].head(4)]).reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(row["path"]).convert("RGB"))
    ax.set_title("IDC" if row["label"] == 1 else "non-IDC")
    ax.axis("off")

plt.tight_layout()
plt.savefig(base_path + "/plots/idc_sample_patches.png", dpi=150, bbox_inches="tight")
plt.show()

### 3. Model — DenseNet-121

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.densenet121(pretrained=True)

model.classifier = nn.Sequential(
    nn.Linear(1024, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 1)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()   # mixed precision

### 4. Training (4 epochs, mixed precision)

In [ ]:
EPOCHS = 4
history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.float().to(device)

        with torch.cuda.amp.autocast():
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item():.4f}")

    # ---------------- validation ----------------
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            val_loss += criterion(outputs, labels).item()

            preds = (torch.sigmoid(outputs) > 0.5).int()
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), base_path + "/models/densenet121_idc_best.pth")
        print("  -> new best model saved")

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], color="#0072B2", lw=2, label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], color="#D55E00", lw=2, label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()

axes[1].plot(epochs_range, history["val_acc"], color="#009E73", lw=2, label="Val accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/densenet121_idc_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Evaluation on the held-out patients

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, accuracy_score)

model.load_state_dict(torch.load(base_path + "/models/densenet121_idc_best.pth"))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(np.array(labels))

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
preds = (all_probs > 0.5).astype(int)

acc = accuracy_score(all_labels, preds)
auc = roc_auc_score(all_labels, all_probs)

print("Test Accuracy:", round(acc, 4))
print("ROC-AUC:", round(auc, 4))
print("\nConfusion Matrix:\n", confusion_matrix(all_labels, preds))
print("\nClassification Report:\n")
print(classification_report(all_labels, preds,
                            target_names=["non-IDC", "IDC"]))

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion matrix (single-hue sequential colormap)
cm = confusion_matrix(all_labels, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["non-IDC", "IDC"],
            yticklabels=["non-IDC", "IDC"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, color="#0072B2", lw=2, label=f"ROC (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Chance")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.tight_layout()
plt.savefig(base_path + "/plots/densenet121_idc_confusion_roc.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Save artifacts to Drive

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
f1 = f1_score(all_labels, preds)

results_text = f"""MODEL: DenseNet-121 (fast setting)
DATASET: IDC breast histopathology patches (patient-wise split)
LOSS: BCEWithLogitsLoss

Test Accuracy: {acc:.4f}
ROC-AUC: {auc:.4f}

Malignant class:
Precision: {precision:.4f}
Recall: {recall:.4f}
F1-score: {f1:.4f}

Confusion Matrix:
{confusion_matrix(all_labels, preds)}

Classification Report:
{classification_report(all_labels, preds)}
"""

with open(base_path + "/results/densenet121_idc_results.txt", "w") as f:
    f.write(results_text)

print(results_text)
print("Results saved")

In [ ]:
results_df = pd.DataFrame({
    "path": test_df["path"].values,
    "true_label": all_labels,
    "predicted_label": preds,
    "probability": all_probs
})

results_df.to_csv(base_path + "/results/densenet121_idc_predictions.csv", index=False)
print("Predictions saved")

In [ ]:
import json

config = {
    "model": "DenseNet-121",
    "pretrained": "ImageNet",
    "dataset": "IDC (paultimothymooney/breast-histopathology-images)",
    "task": "binary classification (IDC vs non-IDC)",
    "split": "patient-wise 70/15/15 + balanced subsample",
    "patches_per_class": {"train": 3000, "val": 750, "test": 750},
    "loss": "BCEWithLogitsLoss",
    "optimizer": "Adam",
    "lr": 1e-4,
    "epochs": 4,
    "batch_size": 64,
    "image_size": 96,
    "mixed_precision": True,
    "threshold": 0.5,
    "seed": 42
}

with open(base_path + "/results/densenet121_idc_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Config saved")

In [ ]:
print(f"Total notebook time: {(time.time() - t_start) / 60:.1f} min")